# 04. PyTorch Custom Datasets Video Notebook

**Resources:**

* Book version of the course materials for 04: https://www.learnpytorch.io/04_pytorch_custom_datasets/

* Ground truth version of notebook 04: https://github.com/mrdbourke/pytorch-deep-learning/blob/main/04_pytorch_custom_datasets.ipynb

## 0 Importing PyTorch and setting up device-agnostic code

In [ ]:
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1 Get data

Our dataset is a subset of the Food101 dataset.

Food101 starts 101 different classes of food and 1000 images per class (750 training, 250 testing).

Our dataset starts with 3 classes of food and only 10% of the images (~75 training, 25 testing).

Why do this?

When starting out ML projects, it's important to try things on a small scale and then increase the scale when necessary. The whole point is to speed up how fast you can experiment. |

In [ ]:
import requests
import zipfile
from pathlib import Path

# Setup path to a data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# if the image folder doesn't exist, download and prepare it...
if image_path.is_dir():
    print(f"{image_path} directoru already exists... skipping download")
else:
    print(f"{image_path} does not exist, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

# Download pizza, steak and sushi data
url = "https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip"

with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    response = requests.get(url)
    print("Downloading...")
    f.write(response.content)

# Unzip
with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping...")
    zip_ref.extractall(image_path)

## 2 Data preparition and exploration

In [ ]:
import os
def walk_through_dir(dir_path):
  """Walks through dir_path returning its contents."""
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

In [ ]:
walk_through_dir(image_path)

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

### 2.1 Visualizing image

Let's write some code to:

1. Get all of the image paths
2. Pick a random image path using Python's random.choice()
3. Get the image class name using `pathlib.Path.parent.stem`
4. Since we're working with images, let's open the image with Python's PIL
5. We'll then show the image and print metada.

In [ ]:
import random
from PIL import Image
# Set seed
#random.seed(42)

# 1. Get all image paths
image_path_list = list(image_path.glob("*/*/*.jpg"))

# 2. Pick a random image path
random_image_path = random.choice((image_path_list))

# 3. Get image class from path name
image_class = random_image_path.parent.stem

# 4. Open image
img = Image.open(random_image_path)

# 5. Print metadata
print(f"Random image path: {random_image_path}")
print(f"Image class: {image_class}")
print(f"Image height: {img.height}")
print(f"Image width: {img.width}")
img

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Turn the image as into an array
img_as_array = np.asarray(img)

# Plot the image
plt.figure(figsize=(10,7));
plt.imshow(img_as_array);
plt.title(f"Image class: {image_class} | Image shape: {img_as_array.shape} -> [height, width, color_channels]");
plt.axis(False);

## 3 Transforming data

Before we can use our image data with PyTorch:

1. Turn your target data into tensors (in our case, numerical representation of our images).

2. Turn it into a `torch.utils.data.Dataset` and subsequently a `torch.utils.data. DataLoader`, we'll call these `Dataset` and `DataLoader`.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

### 3.1 Transforming data with `torchvision.transforms`

In [ ]:
# Write a transform for image
data_transform = transforms.Compose([
    # Resize our images
    transforms.Resize(size=(64,64)),
    # Flip the images randomly on the horizontal
    transforms.RandomHorizontalFlip(p=0.5),
    # Turn the image into tensor
    transforms.ToTensor()
])

In [ ]:
data_transform(img)

In [ ]:
def plot_transformed_images(image_paths, transform, n, seed):
  if seed:
    random.seed(seed)
  random_image_paths = random.sample(image_paths, k=n)
  for image_path in random_image_paths:
    with Image.open(image_path) as f:
      fig, ax = plt.subplots(nrows=1, ncols=2)
      ax[0].imshow(f)
      ax[0].set_title(f"Original\nSize: {f.size}")
      ax[0].axis(False)

      # Transform and plot target
      transformed_image = transform(f).permute(1,2,0) # for matplotlib H W C
      ax[1].imshow(transformed_image)
      ax[1].set_title(f"Transformed\nShape: {transformed_image.shape}")
      ax[1].axis(False)

      fig.suptitle(f"Class: {image_path.parent.stem}", fontsize=16)

In [ ]:
plot_transformed_images(image_paths=image_path_list,
                        transform=data_transform,
                        n=3,
                        seed=46)

## 4 Option 1: Loading image data using `ImageFolder`

We can load image classification data using `torchvision.datasets.ImageFolder`

In [ ]:
# Use ImageFolder to create datasets
from torchvision import datasets
train_data = datasets.ImageFolder(root=train_dir,
                                  transform=data_transform,
                                  target_transform=None)
test_data = datasets.ImageFolder(root=test_dir,
                                 transform=data_transform,
                                 target_transform=None)
train_data, test_data

In [ ]:
class_names = train_data.classes
class_names

In [ ]:
class_dict = train_data.class_to_idx
class_dict

In [ ]:
img, label = train_data[0][0], train_data[0][1]
img

In [ ]:
label

### 4.1 Turn loaded images into `DataLoader`

In [ ]:
from torch.utils.data import DataLoader
train_data_loader = DataLoader(batch_size=1,
                               shuffle=True,
                               dataset=train_data,
                               num_workers=1)
test_data_loader = DataLoader(batch_size=1,
                              num_workers=1,
                              shuffle=False,
                              dataset=test_data)
train_data_loader, test_data_loader

In [ ]:
img, label = next(iter(train_data_loader))
# Batch size will now be 1, you can change the batch size if you like
print(f"Image shape: {img.shape} -> [batch_size, color_channels, height, width]")
print(f"Label shape: {label.shape}")

## 5 Option 2: Loading Image Data with a Custom `Dataset`

1. Want to be able to load images from file
2. Want to be able to get class names from the Dataset
3. Want to be able to get classes as dictionary from the Dataset

Pros:
* Can create a `Dataset` out of almost anything
* Not limited to PyTorch pre-built `Dataset` functions

Cons:
* Even though you could create `Dataset` out of almost anything, it doesn't mean it will work ...
* Using a custom `Dataset` often results in us writing more code, which could be prone to errors or performance issues

In [ ]:
from typing import Tuple, Dict, List

### 5.1 Creating a helper function to get class names

We want a function to: 1. Get the class names using `os.scandir()` to traverse a target directory (ideally the directory is in standard image classification format).
2. Raise an error if the class names aren't found (if this happens, there might be something wrong with the directory structure).
3. Turn the class names into a dict and a list and return them.

In [ ]:
# Setup path for target directory
target_directory = train_dir
print(f"Target dir: {target_directory}")

# Get the class names from the target directory
class_names_found = sorted([entry.name for entry in list(os.scandir(target_directory))])
class_names_found

In [ ]:
def find_classes(directory: str):
  """Finds the class folder names in a target directory."""
  # 1. Get the class names by scanning the target directory
  classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())

  # 2. Raise an error if class names could not be found
  if not classes:
    raise FileNotFoundError(f"Couldn't find any classes in {directory}")

  # 3. Create a dictionary of index label
  class_to_idx = {class_name: i for i, class_name in enumerate(classes)}
  return classes, class_to_idx

In [ ]:
find_classes(target_directory)

### 5.2 Create a custom Dataset to replicate ImageFolder

To create our own custom dataset, we want to:
1. Subclass `torch. utils.data.Dataset`
2. Init our subclass with a target directory (the directory we'd like to get data from) as well as a transform if we'd like to transform our data.
3. Create several attributes:
* paths - paths of our images
* transform - the transform we'd like to use
* classes - a list of the target classes
* class_to_idx - a dict of the target classes mapped to integer labels
4. Create a function to `load_images()`, this function will open an image
5. Overwrite the `__len()__` method to return the length of our dataset
6. Overwrite the `__getitem()__` method to return a given sample when passed an index

In [ ]:
from torch.utils.data import Dataset

# 1. Subclass torch.utils.data.Dataset
class ImageFolderCustom(Dataset):
  # 2. Initialize our custom dataset
  def __init__(self,
               targ_dir,
               transform=None):
    # 3. Create class attributes
    self.paths = list(pathlib.Path(targ_dir).glob("*/*.jpg"))
    self.transform = transform
    self.classes, self.class_to_idx = find_classes(targ_dir)

  # 4. Create a function to load images
  def load_image(self, index):
    """Opens an image via a path and returns it."""
    image_path = self.paths(index)
    return Image.open(image_path)

  # 5. Overwrite __len__()
  def __len__(self):
    """Returns the total number of samples."""
    return len(self.paths)

  # 6. Overwrite __getitem__() method to return a particular sample
  def __getitem__(self, index):
    """Returns one sample of data, data and label (X, y)."""
    img = self.load_image()
    class_name = self.paths[index].parent.name
    class_idx = self.classes_to_idx[class_name]

    if self.transform:
      return self.transform(img), class_idx
    else:
      return img, class_idx

## 6 Other forms of transforms (data augmentation)

Data augmentation is the process of artificially adding diversity to your training data. In the case of image data, this may mean applying various image transformations to the training images.
This practice hopefully results in a model that's more generalizable to unseen data.
Let's take a look at one particular type of data augmentation used to train PyTorch vision models to state of the art levels ... Blog post: https://pytorch.org/blog/how-to-train-state-of-the-art-models-using-torchvision-latest-primitives/#break-down-of-key-accuracy-improvements

In [ ]:
train_transform = transforms.Compose([transforms.Resize(size=(224, 224)),
                                      transforms.TrivialAugmentWide(num_magnitude_bins=31),
                                      transforms. ToTensor()])

test_transform = transforms.Compose([transforms.Resize(size=(224, 224)),
                                     transforms.ToTensor()])

In [ ]:
plot_transformed_images(
    image_paths=image_path_list,
    transform = train_transform,
    n=3,
    seed=None
)

## 7 Model 0: TinyVGG without data augmantation

Let's replicate TinyVGG architecture from the CNN Explanier

### 7.1 Creating transform and loading data for Model 0

In [ ]:
# Create simple transform
simple_transform = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.ToTensor()
])

In [ ]:
# 1. Load and transform data
train_data_simple = datasets.ImageFolder(root=train_dir,
                                         transform=simple_transform)
test_data_simple = datasets.ImageFolder(root=test_dir,
                                        transform=simple_transform)

# 2.Turn the datasets into DataLoaders

train_dataloader_simple = DataLoader(dataset=train_data_simple,
                                     batch_size=32,
                                     num_workers=2,
                                     shuffle=True)
test_dataloader_simple = DataLoader(dataset=test_data_simple,
                                    batch_size=32,
                                    num_workers=2,
                                    shuffle=False)

### 7.2 Create TinyVGG model class


In [ ]:
class TinyVGG(nn.Module):
  def __init__(self,input_shape,hidden_units,output_shape):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )
    self.classifier = nn.Sequential(
          nn.Flatten(),
          nn.Linear(in_features=hidden_units*13*13,
                    out_features=output_shape)
    )

  def forward(self,x):
    x = self.conv_block_1(x)
    #print(x.shape)
    x = self.conv_block_2(x)
    #print(x.shape)
    x = self.classifier(x)
    #print(x.shape)
    return x

In [ ]:
model_0 = TinyVGG(input_shape=3,
                  hidden_units=10,
                  output_shape=len(class_names))

### 7.3 Try a forward pass on a singel image (to test the model)

In [ ]:
/

In [ ]:
model_0(image_batch)

### 7.4 Use `torchinfo` to get an idea of the shapes going through our model

In [ ]:
#import torchinfo
#from torchinfo import summary
#summary(model_0, input_size=(1,3,64,64))

### 7.5 Create train and test loop functions


In [ ]:
# Create train steps()
def train_step(model,
              dataloader,
              loss_fn,
              optimizer):

  model.train()

  train_loss, train_acc = 0, 0

  for batch, (X, y) in enumerate(dataloader):

    y_pred = model(X)

    loss = loss_fn(y_pred, y)
    train_loss += loss.item()

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
    train_acc += (y_pred_class==y).sum().item()/len(y_pred)

  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader) *100
  return train_loss, train_acc

In [ ]:
# Create a test step()
def test_step(model,
              dataloader,
              loss_fn):
  model.eval()

  test_loss, test_acc = 0, 0
  with torch.inference_mode():
    for batch, (X,y) in enumerate(dataloader):
      test_pred_logits = model(X)

      loss = loss_fn(test_pred_logits,y)
      test_loss += loss.item()

      test_pred_labels = test_pred_logits.argmax(dim=1)
      test_acc += (test_pred_labels==y).sum().item()/len(test_pred_labels)
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader) *100
    return test_loss, test_acc

### 7.6 Creating a `train()` function to combine `train_step()` and `test_step()`

In [ ]:
def train(model,
          train_data,
          test_data,
          loss_fn,
          optimizer,
          epochs=5):

  results = {"train_loss": [],
           "train_acc": [],
           "test_loss":[],
           "test_acc":[]}
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model=model,
                                       dataloader=train_dataloader_simple,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer)
    test_loss, test_acc = test_step(model=model,
                                    dataloader=test_dataloader_simple,
                                    loss_fn=loss_fn,)

    print(f"Epoch: {epoch} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.2f}% | Test loss: {test_loss:.4f} | Test acc: {test_acc:.2f}%")

    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  return results

In [ ]:
from timeit import default_timer as timer
from tqdm import tqdm

### 7.7 Train and avaluate model 0

In [ ]:
# Set random seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set epochs

NUM_EPOCHS = 10

# Recreate an instance of TinyVGG
model_0 = TinyVGG(input_shape=3,
                  hidden_units=10,
                  output_shape=len(train_data.classes))

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(),
                             lr=0.001)

# Start the timer
start_time = timer()

# Train model_0
model_0_results = train(model=model_0,
                        train_data=train_dataloader_simple,
                        test_data=test_dataloader_simple,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS)

# End the timer
end_time = timer()
print(f"Total trainin time: {end_time-start_time:.3f} seconds")

### 7.8 Plot the loss curves of Model 0

A **loss curve** is a way of tracking your model's progress over time

In [ ]:
# Get the model_0_results keys
model_0_results.keys()

In [ ]:
def plot_loss_curves(results):
  # Get the loss values of the results dictionary
  loss = results["train_loss"]
  test_loss = results["test_loss"]

  # Get the accuracy values of the results dictionary
  accuracy = results["train_acc"]
  test_accuracy = results["test_acc"]

  # Figure out how many epochs there were
  epochs = range(len(results["train_loss"]))

  # Setup a plot
  plt.figure(figsize=(15,7))

  # Plot the loss
  plt.subplot(1,2,1)
  plt.plot(epochs, loss, label="train_loss")
  plt.plot(epochs, test_loss, label="test_loss")
  plt.title("Loss")
  plt.xlabel("Epochs")
  plt.legend();

  # Plot the accuracy
  plt.subplot(1,2,2)
  plt.plot(epochs, accuracy, label="train_accuracy")
  plt.plot(epochs, test_accuracy, label="test_accuracy")
  plt.title("Accuracy")
  plt.xlabel("Epochs")
  plt.legend();


In [ ]:
plot_loss_curves(results=model_0_results)

## 8 Model 1: TinyVGG with data Augmentation

### 8.1 Create transform with data augmentation

In [ ]:
# Create training transform with TrivialAugment
train_transform_trivial = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor()
])

test_transform_simple = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.ToTensor()
])

### 8.2 Create train and test Datasets and Dataloader with augmentation

In [ ]:
# Turn image folders into Datasets
train_data_augmented = datasets.ImageFolder(root=train_dir,
                                            transform=train_transform_trivial)
test_data_simple = datasets.ImageFolder(root=test_dir,
                                        transform=test_transform_simple)

In [ ]:
train_dataloader_augmented = DataLoader(dataset=train_data_augmented,
                                        batch_size=16,
                                        shuffle=True,
                                        num_workers=2)
test_dataloader_simple = DataLoader(dataset=test_data_simple,
                                    shuffle=False,
                                    batch_size=16,
                                    num_workers=2)

In [ ]:
class TinyVGGV1(nn.Module):
    def __init__(self, input_shape, output_shape):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(input_shape, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, output_shape)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x


In [ ]:
#class TinyVGGV1(nn.Module):
  def __init__(self,input_shape,hidden_units,output_shape):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     padding=0,
                     stride=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     padding=0,
                     stride=2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*16*16,
                  out_features=hidden_units*16*16),
        nn.ReLU(),
        nn.Linear(in_features=hidden_units*16*16,
                  out_features=output_shape)
    )
  def forward(self,x):
    return self.classifier(self.conv_block_2(self.conv_block_1(x)))

In [ ]:
model_1 = TinyVGGV1(input_shape=3,
                    output_shape=len(class_names))

In [ ]:
image_batch, label_batch = next(iter(train_dataloader_augmented))
image_batch.shape, label_batch.shape

In [ ]:
model_1(image_batch)

In [ ]:
model_1_results = train(train_data=train_data_augmented,
      test_data=test_dataloader_simple,
      loss_fn=nn.CrossEntropyLoss(),
      optimizer=torch.optim.Adam(params=model_1.parameters(),lr=0.001),
      model=model_1,
      epochs=10)

In [ ]:
plot_loss_curves(model_1_results)

In [ ]:
import pandas as pd
model_0_pd = pd.DataFrame(model_0_results)
model_1_pd = pd.DataFrame(model_1_results)

In [ ]:
plt.figure(figsize=(15,10))

epochs = range(len(model_0_pd))

plt.subplot(2,2,1)
plt.plot(epochs, model_0_pd["train_loss"], label="Model 0")
plt.plot(epochs, model_1_pd["train_loss"], label="Model 1")
plt.title("Train Loss")
plt.xlabel("Epochs")
plt.legend();

plt.subplot(2,2,2)
plt.plot(epochs, model_0_pd["test_loss"], label="Model 0")
plt.plot(epochs, model_1_pd["test_loss"], label="Model 1")
plt.title("Test Loss")
plt.xlabel("Epochs")
plt.legend();

plt.subplot(2,2,3)
plt.plot(epochs, model_0_pd["train_acc"], label="Model 0")
plt.plot(epochs, model_1_pd["train_acc"], label="Model 1")
plt.title("Train Accuracy")
plt.xlabel("Epochs")
plt.legend();

plt.subplot(2,2,4)
plt.plot(epochs, model_0_pd["test_acc"], label="Model 0")
plt.plot(epochs, model_1_pd["test_acc"], label="Model 1")
plt.title("Test Accuracy")
plt.xlabel("Epochs")
plt.legend();

## 9 Making a prediction on a custom image

In [ ]:
# Download custom image
import requests

# Setup custom image path
custom_image_path = data_path / "04-pizza-dad.jpeg"

# Download the image if it doesn't already exist
if not custom_image_path.is_file():
  with open(custom_image_path, "wb") as f:
    # When downloading from GitHub, need to use the "raw" file link
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg")
    print(f"Downloading {custom_image_path}...")
    f.write(request.content)
else:
 print(f"{custom_image_path} already exists, skipping download ... ")

### 11.1 Loading in a custom image with PyTorch
We have to make sure our custom image is in the same format as the data our model was trained on.
* In tensor form with datatype (torch.float32)
* Of shape 64x64x3
* On the right device

We can read an image into PyTorch using - https://pytorch.org/vision/stable/generated/torchvision.io.read_image.html#torchvision.io.read_image

In [ ]:
import torchvision
custom_image_unit8 = torchvision.io.read_image(str(custom_image_path)).type(torch.float32) / 255
custom_image_unit8.shape, custom_image_unit8.dtype

In [ ]:
plt.imshow(custom_image_unit8.permute(1,2,0))

In [ ]:
custom_image_transform = transforms.Compose([
    transforms.Resize(size=(64,64)),
])
custom_image_transformed = custom_image_transform(custom_image_unit8)
custom_image_transformed.shape

In [ ]:
plt.imshow(custom_image_transformed.permute(1,2,0))

In [ ]:
#custom_image_transformed = custom_image_transformed.unsqueeze(0)
model_1.eval()
with torch.inference_mode():
  custom_pred = model_1(custom_image_transformed)

In [ ]:
custom_pred

Note, to make a prediction on a custom image we had to:
* Load the image and turn it into a tensor
* Make sure the image was the same datatype as the model (torch.float32)
* Make sure the image was the same shape as the data the model was trained on (3, 64, 64) with a batch size ... (1, 3, 64, 64)
* Make sure the image was on the same device as our model

In [ ]:
custom_pred_probs = torch.softmax(custom_pred,dim=1)
custom_pred_probs

In [ ]:
the_pred = class_names[custom_pred_probs.argmax(dim=1)]
the_pred

## 10 Make a function to putting all together

In [ ]:
def tahmin(url, transforms, file_name: str, model):
  # Setup custom image path
  custom_image_path = data_path / file_name

  # Download the image if it doesn't already exist
  if not custom_image_path.is_file():
    with open(custom_image_path, "wb") as f:
      # When downloading from GitHub, need to use the "raw" file link
      request = requests.get(url)
      print("Image downloaded")
      f.write(request.content)

  custom_image_unit8 = torchvision.io.read_image(str(custom_image_path)).type(torch.float32) / 255
  #print(f"Shape: {custom_image_unit8.shape}")
  #print(f"Type: {custom_image_unit8.dtype}")

  custom_image_transformed = transforms(custom_image_unit8)
  #print(f"Shape: {custom_image_transformed.shape}")

  model_1.eval()
  with torch.inference_mode():
    custom_pred = model_1(custom_image_transformed.unsqueeze(dim=0))
  #print(f"Shape: {custom_image_transformed.shape}")

  custom_pred_probs = torch.softmax(custom_pred,dim=1)

  the_pred = class_names[custom_pred_probs.argmax(dim=1)]

  return the_pred

In [ ]:
b = tahmin(url="https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg", transforms=custom_image_transform, model=model_0, file_name="image.jpeg")
print(f"Ilk tahminim: {b}")

In [ ]:
a = tahmin(url="https://media.istockphoto.com/id/1042948900/tr/foto%C4%9Fraf/beyaz-arka-plan-%C3%BCzerinde-izole-pizza-sucuk.jpg?s=1024x1024&w=is&k=20&c=LEfl5akh3N9nFKaOQyJyqL6p9aIKXdqVG0tUCn10mQM=", file_name="pizza.jpg", model=model_1, transforms=custom_image_transform)
print(f"Son tahminim: {a}")

In [ ]:
d = tahmin(url="https://image.milimaj.com/i/milliyet/75/0x0/6938154bee16fd6f938bf4b4.jpg", file_name="sus5i.jpeg", model=model_1, transforms=custom_image_transform)
print(f"VEEE {d}")